In [19]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
from pathlib import Path
import hashlib
import json
import re
import unicodedata
import platform
from collections import Counter
from datetime import datetime, timezone
from itertools import combinations
import pandas as pd

PROJECT = Path('/content/drive/MyDrive/text_mining_project_01')
ORIGINAL = PROJECT / 'original_data'
CLEAN = PROJECT / 'clean_data'

ARQUIVOS = ['train.csv', 'valid.csv', 'test.csv', 'sample_submission.csv']
ALVOS = ['formal_register', 'thematic_coherence',
         'narrative_rhetorical_structure', 'cohesion']
TAMANHOS_ESPERADOS = {'train': 740, 'valid': 125, 'test': 370}

if not ORIGINAL.is_dir():
    raise FileNotFoundError(
        'Pasta não encontrada. Confira o nome e a localização no Drive: '
        f'{ORIGINAL}'
    )

CLEAN.mkdir(exist_ok=True)
print('Arquivos encontrados:')
for arquivo in sorted(ORIGINAL.iterdir()):
    print(arquivo.name)

faltantes = [nome for nome in ARQUIVOS if not (ORIGINAL / nome).is_file()]
if faltantes:
    raise FileNotFoundError(f'Arquivos ausentes em {ORIGINAL}: {faltantes}')
print('Arquivos localizados:', ARQUIVOS)


Arquivos encontrados:
sample_submission.csv
test.csv
train.csv
valid.csv
Arquivos localizados: ['train.csv', 'valid.csv', 'test.csv', 'sample_submission.csv']


In [21]:
hashes_originais = {
    nome: hashlib.sha256((ORIGINAL / nome).read_bytes()).hexdigest()
    for nome in ARQUIVOS
}
brutos = {
    nome: pd.read_csv(ORIGINAL / f'{nome}.csv', encoding='utf-8-sig',
                     dtype=str, keep_default_na=False)
    for nome in ['train', 'valid', 'test']
}
sample = pd.read_csv(ORIGINAL / 'sample_submission.csv',
                     encoding='utf-8-sig', dtype=str, keep_default_na=False)
for nome, df in brutos.items():
    print(nome, df.shape, df.columns.tolist())
display(brutos['train'].head(3))


train (740, 7) ['id', 'essay', 'prompt', 'formal_register', 'thematic_coherence', 'narrative_rhetorical_structure', 'cohesion']
valid (125, 7) ['id', 'essay', 'prompt', 'formal_register', 'thematic_coherence', 'narrative_rhetorical_structure', 'cohesion']
test (370, 3) ['id', 'essay', 'prompt']


,id,essay,prompt,formal_register,thematic_coherence,narrative_rhetorical_structure,cohesion
0,753,[T] Os vidros de tintas\nOs vridos de tintas p...,Eu encontrei em cima do armário alguns potes c...,3,1,1,1
1,582,[T] O ARMÁRIO E TINTA MAGICA\nEU ENCONTREI CON...,Eu encontrei em cima do armário alguns potes c...,2,2,3,2
2,548,[T] Li um livro que me assustou !\n[P] Em uma ...,Eu encontrei em cima do armário alguns potes c...,3,1,5,3


In [22]:
dados = {
    nome: df.copy(deep=True)
    for nome, df in brutos.items()
}

resumo = []
problemas = []

for nome, df in dados.items():
    colunas_esperadas = ["id", "essay", "prompt"]

    if nome in ["train", "valid"]:
        colunas_esperadas += ALVOS

    faltam = set(colunas_esperadas) - set(df.columns)
    extras = set(df.columns) - set(colunas_esperadas)

    if faltam or extras:
        problemas.append(
            f"{nome}: colunas ausentes={sorted(faltam)}, "
            f"colunas extras={sorted(extras)}"
        )

    if len(df) != TAMANHOS_ESPERADOS[nome]:
        problemas.append(
            f"{nome}: esperadas {TAMANHOS_ESPERADOS[nome]} "
            f"linhas, encontradas {len(df)}."
        )

    if faltam:
        continue

    ids_vazios = df["id"].str.strip().eq("")
    ids_duplicados = df["id"].duplicated(keep=False)

    textos_vazios = df["essay"].str.strip().eq("")
    temas_vazios = df["prompt"].str.strip().eq("")

    if ids_vazios.any():
        problemas.append(f"{nome}: existem IDs vazios.")

    if ids_duplicados.any():
        problemas.append(f"{nome}: existem IDs repetidos.")

    # Validar antes de converter as notas para números
    if nome in ["train", "valid"]:
        for alvo in ALVOS:
            notas = pd.to_numeric(df[alvo], errors="coerce")
            invalidas = ~notas.isin([1, 2, 3, 4, 5])

            if invalidas.any():
                problemas.append(
                    f"{nome}: {int(invalidas.sum())} "
                    f"notas inválidas na coluna {alvo}."
                )

                display(df.loc[invalidas, ["id", alvo]])
            else:
                df[alvo] = notas.astype("int64")

    resumo.append({
        "conjunto": nome,
        "linhas": len(df),
        "colunas": len(df.columns),
        "ids_vazios": int(ids_vazios.sum()),
        "linhas_com_id_repetido": int(ids_duplicados.sum()),
        "redacoes_vazias": int(textos_vazios.sum()),
        "temas_vazios": int(temas_vazios.sum()),
        "linhas_duplicadas_excedentes": int(
            df.duplicated().sum()
        ),
        "redacoes_duplicadas_excedentes": int(
            df["essay"].duplicated().sum()
        ),
    })

diagnostico = pd.DataFrame(resumo)
display(diagnostico)

if problemas:
    raise ValueError(
        "Revise os problemas encontrados:\n- "
        + "\n- ".join(problemas)
    )

print("Validação estrutural concluída.")
print("Nenhuma linha foi removida.")

,conjunto,linhas,colunas,ids_vazios,linhas_com_id_repetido,redacoes_vazias,temas_vazios,linhas_duplicadas_excedentes,redacoes_duplicadas_excedentes
0,train,740,7,0,0,0,0,0,0
1,valid,125,7,0,0,0,0,0,0
2,test,370,3,0,0,0,0,0,0


Validação estrutural concluída.
Nenhuma linha foi removida.


In [23]:
# Verificar se algum ID aparece em conjuntos diferentes
for conjunto_a, conjunto_b in combinations(dados.keys(), 2):
    ids_comuns = (
        set(dados[conjunto_a]["id"])
        & set(dados[conjunto_b]["id"])
    )

    if ids_comuns:
        raise ValueError(
            f"IDs compartilhados entre {conjunto_a} "
            f"e {conjunto_b}: {sorted(ids_comuns)}"
        )

print("Não há IDs compartilhados entre os conjuntos.")


# Normalização para comparar redações
def chave_comparacao(texto):
    texto = unicodedata.normalize("NFC", texto)
    return re.sub(r"\s+", " ", texto).strip()


auditoria = pd.concat(
    [
        df[["id", "essay"]].assign(conjunto=nome)
        for nome, df in dados.items()
    ],
    ignore_index=True,
)

auditoria["chave"] = auditoria["essay"].map(
    chave_comparacao
)

mascara = (
    auditoria["chave"].ne("")
    & auditoria["chave"].duplicated(keep=False)
)

repetidas = auditoria.loc[mascara].copy()

repetidas["grupo"] = (
    pd.factorize(repetidas["chave"])[0] + 1
)

repetidas["entre_conjuntos"] = (
    repetidas.groupby("grupo")["conjunto"]
    .transform("nunique")
    .gt(1)
)

duplicatas = repetidas[
    ["grupo", "conjunto", "id", "entre_conjuntos"]
].sort_values(["grupo", "conjunto", "id"])

if duplicatas.empty:
    print("Nenhuma redação repetida nessa comparação.")
else:
    display(duplicatas)
    print("Casos registrados para revisão. Nenhuma linha removida.")

Não há IDs compartilhados entre os conjuntos.


,grupo,conjunto,id,entre_conjuntos
1086,1,test,761,True
714,1,train,760,True


Casos registrados para revisão. Nenhuma linha removida.


In [24]:
MARCADORES = {
    "paragrafo": ["[P]", "[ P]", "[P}", "[p]", "{p}"],
    "titulo": ["[T]", "[t]", "{t}"],
    "rasura": [
        "[R]", "[X]", "[X~]", r"[X\~]",
        "[r]", "[x]", "{x}",
    ],
    "simbolo": ["[S]", "[s]"],
    "desconhecido": ["[?]", "{?}", "[?}", "{?]"],
    "fora_da_linha": ["[LC]", "[LT]", "[lt]"],
}

TOKEN_GRUPO = {
    token: grupo
    for grupo, tokens in MARCADORES.items()
    for token in tokens
}

PADROES = {
    grupo: re.compile(
        "|".join(
            re.escape(token)
            for token in sorted(tokens, key=len, reverse=True)
        )
    )
    for grupo, tokens in MARCADORES.items()
}

# Busca possíveis marcações curtas entre colchetes,
# chaves ou sinais de < >
CANDIDATOS = re.compile(
    r"[\[{][^\[\]{}\n]{0,30}[\]}]|<[^<>\n]{1,30}>"
)

registros = []

for nome, df in dados.items():
    contagens = Counter(
        token
        for texto in df["essay"]
        for token in CANDIDATOS.findall(texto)
    )

    for token, quantidade in sorted(contagens.items()):
        registros.append({
            "conjunto": nome,
            "token": token,
            "ocorrencias": quantidade,
            "grupo": TOKEN_GRUPO.get(
                token, "nao_documentado"
            ),
        })

inventario = pd.DataFrame(
    registros,
    columns=["conjunto", "token", "ocorrencias", "grupo"],
)

print("Inventário completo:")
display(inventario)

print("Marcações não documentadas — serão preservadas:")
display(
    inventario.loc[
        inventario["grupo"].eq("nao_documentado")
    ]
)

Inventário completo:


,conjunto,token,ocorrencias,grupo
0,train,<i>,8,nao_documentado
1,train,<t>,9,nao_documentado
2,train,[ P],1,paragrafo
3,train,[?],822,desconhecido
4,train,[?},3,desconhecido
5,train,[LC],49,fora_da_linha
6,train,[LT],291,fora_da_linha
7,train,[P],1006,paragrafo
8,train,[P},1,paragrafo
9,train,[S],88,simbolo


Marcações não documentadas — serão preservadas:


,conjunto,token,ocorrencias,grupo
0,train,<i>,8,nao_documentado
1,train,<t>,9,nao_documentado
12,train,[X},2,nao_documentado
23,valid,<i>,2,nao_documentado
37,test,<i/>,1,nao_documentado
38,test,<i>,1,nao_documentado
39,test,<t>,2,nao_documentado
48,test,[X},1,nao_documentado


In [25]:
def limpar_texto(texto):
    # Padroniza Unicode sem remover acentos
    texto = unicodedata.normalize("NFC", texto)

    # Padroniza as quebras de linha
    texto = texto.replace("\r\n", "\n").replace("\r", "\n")

    # Substitui somente os marcadores documentados
    for grupo, padrao in PADROES.items():
        substituto = "\n" if grupo == "paragrafo" else " "
        texto = padrao.sub(substituto, texto)

    # Reduz espaços e tabulações sem apagar quebras de linha
    texto = re.sub(r"[^\S\n]+", " ", texto)

    # Retira espaços nas bordas das linhas
    texto = re.sub(r" *\n *", "\n", texto)

    return texto.strip()


def preparar_dataset(df):
    resultado = df.copy(deep=True)

    # Conta as marcações antes de fazer a limpeza
    for grupo, padrao in PADROES.items():
        resultado[f"n_marcadores_{grupo}"] = (
            df["essay"].map(
                lambda texto: len(padrao.findall(texto))
            )
        )

    resultado["n_marcadores_nao_documentados"] = (
        df["essay"].map(
            lambda texto: sum(
                token not in TOKEN_GRUPO
                for token in CANDIDATOS.findall(texto)
            )
        )
    )

    resultado["essay_clean"] = df["essay"].map(limpar_texto)

    resultado["texto_limpo_vazio"] = (
        resultado["essay_clean"].eq("")
    )

    return resultado


limpos = {
    nome: preparar_dataset(df)
    for nome, df in dados.items()
}

print("Versões limpas criadas na memória.")
print("Nenhum arquivo foi sobrescrito.")

Versões limpas criadas na memória.
Nenhum arquivo foi sobrescrito.


In [26]:
resumo_limpeza = []

for nome, df_limpo in limpos.items():
    df_original = dados[nome]

    # Conferir colunas originais
    pd.testing.assert_frame_equal(
        df_limpo[df_original.columns],
        df_original,
    )

    assert df_limpo["essay_clean"].equals(
        df_limpo["essay_clean"].map(limpar_texto)
    ), f"{nome}: a limpeza mudou ao ser aplicada novamente."

    # Verificar marcadores documentados
    for grupo, padrao in PADROES.items():
        restantes = df_limpo["essay_clean"].map(
            lambda texto: bool(padrao.search(texto))
        )

        assert not restantes.any(), (
            f"{nome}: restaram marcadores do grupo {grupo}."
        )

    resumo_limpeza.append({
        "conjunto": nome,
        "linhas_preservadas": len(df_limpo),
        "textos_alterados": int(
            df_limpo["essay_clean"].ne(df_limpo["essay"]).sum()
        ),
        "textos_vazios_apos_limpeza": int(
            df_limpo["texto_limpo_vazio"].sum()
        ),
    })

display(pd.DataFrame(resumo_limpeza))

# Visualizar exemplos
for _, linha in limpos["train"].head(3).iterrows():
    print("\n" + "=" * 60)
    print("ID:", linha["id"])

    print("\nORIGINAL:")
    print(linha["essay"])

    print("\nLIMPO:")
    print(linha["essay_clean"])

,conjunto,linhas_preservadas,textos_alterados,textos_vazios_apos_limpeza
0,train,740,732,0
1,valid,125,120,0
2,test,370,363,0



ID: 753

ORIGINAL:
[T] Os vidros de tintas
Os vridos de tintas pinta livros
Os vridos de tintas pinta parede
Os vridos de tintas pinta onibus
Os vridos de tintas pinta papelão
Os vridos de tintas pinta papel
Os vridos de tintas pinta jogos
Os vridos de tintas pinta brinquedos
Os vridos de tintas desenha
Os vridos de tintas pinta maquinas
Os vridos de tintas pinta bones
Os vridos de tintas pinta bolas
Os vridos de tintas pinta quadros
Os vridos de tintas pinta mesas
Os vridos de tintas pinta materias escolares
Os vridos de tintas pinta [?] de fios de tomadas
Os vridos de tintas pinta cadeiras
Os vridos de tintas pinta chinelos
Os vridos de tintas pinta escolas

LIMPO:
Os vidros de tintas
Os vridos de tintas pinta livros
Os vridos de tintas pinta parede
Os vridos de tintas pinta onibus
Os vridos de tintas pinta papelão
Os vridos de tintas pinta papel
Os vridos de tintas pinta jogos
Os vridos de tintas pinta brinquedos
Os vridos de tintas desenha
Os vridos de tintas pinta maquinas
Os vri

In [27]:
from zoneinfo import ZoneInfo

for nome, hash_original in hashes_originais.items():
    hash_atual = hashlib.sha256(
        (ORIGINAL / nome).read_bytes()
    ).hexdigest()

    if hash_atual != hash_original:
        raise ValueError(
            f"O arquivo original {nome} mudou desde o carregamento."
        )

# Cria uma pasta de versão com data e hora
now = datetime.now(ZoneInfo("America/Recife"))

VERSAO = now.strftime(
    "clean-data_%d-%m-%Y_%H-%M-%S_%f_UTC%z"
)

SAIDA = CLEAN / VERSAO
RELATORIOS = SAIDA / "relatorios"

RELATORIOS.mkdir(parents=True, exist_ok=False)

# Salvar e reler cada CSV
for nome, df in limpos.items():
    destino = SAIDA / f"{nome}_limpo.csv"

    df.to_csv(
        destino,
        index=False,
        encoding="utf-8",
    )

    conferido = pd.read_csv(
        destino,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8",
    )

    pd.testing.assert_frame_equal(
        conferido,
        df.astype(str),
    )

    print(f"Salvo e verificado: {destino.name}")

# Salva os diagnósticos produzidos nos blocos anteriores
relatorios = {
    "diagnostico": diagnostico,
    "duplicatas_originais": duplicatas,
    "inventario_marcadores": inventario,
    "resumo_limpeza": pd.DataFrame(resumo_limpeza),
}

for nome, tabela in relatorios.items():
    tabela.to_csv(
        RELATORIOS / f"{nome}.csv",
        index=False,
        encoding="utf-8",
    )

# Registra a versão dos arquivos e as regras aplicadas
manifesto = {
    "versao": VERSAO,
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "hashes_originais": hashes_originais,
    "linhas_exportadas": {
        nome: len(df)
        for nome, df in limpos.items()
    },
    "marcadores_documentados": MARCADORES,
    "marcadores_nao_documentados": "preservados",
    "coluna_original": "essay",
    "coluna_preparada": "essay_clean",
    "linhas_removidas": 0,
    "correcao_ortografica": False,
    "data_augmentation": False,
}

(RELATORIOS / "manifesto.json").write_text(
    json.dumps(manifesto, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\nExportação concluída.")
print("Pasta da entrega:", SAIDA)

Salvo e verificado: train_limpo.csv
Salvo e verificado: valid_limpo.csv
Salvo e verificado: test_limpo.csv

Exportação concluída.
Pasta da entrega: /content/drive/MyDrive/text_mining_project_01/clean_data/clean-data_14-09-2026_22-03-20_087693_UTC-0300
